# **Track Network - TMV**

### Data Fetching

In [29]:
import pandas as pd
import psycopg2

def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    """
    Connects to PostgreSQL and loads the given table into a Pandas DataFrame.
    """
    try:
        # Connect to PostgreSQL
        connection = psycopg2.connect(
            host=host_ip,
            database=database_name,
            user=user,
            password=password,
            port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")

        # Create query
        query = f"SELECT * FROM {table_name};"

        # Load into pandas DataFrame
        df = pd.read_sql_query(query, connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")

        return df

    except Exception as e:
        print(f"❌ Error: {e}")
        return None

    finally:
        if connection:
            connection.close()

# --- Configuration (same as before) ---
HOST_IP = "100.95.110.69"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

Connected successfully to pradigma-extractor on 100.95.110.69


C:\Users\win 11\AppData\Local\Temp\ipykernel_12484\3393819530.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, connection)


✅ Fetched 7760 rows from 'extraction'


In [30]:
keywords = ["TMV", "TMV3"]

pattern = '|'.join(keywords)

df = df_original.copy(deep=True)
df = df[
    (df['status_id'] == 1) &
    (df['dept_name'] == 'Track-Network') &
    (df['filename'].str.contains(pattern, case=False, na=False))
][['filename', 'workorder_id', 'json_data']]

df.head(5)


,filename,workorder_id,json_data
2000,TN_PM_MTH_TMV1_4000583484.pdf,4.000583e+09,"{'notification': {'notification_no': 'NA', 'no..."
2008,TN_PM_MTH_TMV1_4000588662.pdf,4.000589e+09,"{'notification': {'notification_no': 'NA', 'no..."
2021,TN_PM_MTH_TMV1_4000597623.pdf,4.000598e+09,"{'notification': {'notification_no': 'NA', 'no..."
2054,TN_PM_MTH_TMV1_4000619061.pdf,4.000619e+09,"{'notification': {'notification_no': 'NA', 'no..."
2061,TN_PM_MTH_TMV1_4000626055.pdf,4.000626e+09,"{'notification': {'notification_no': 'NA', 'no..."


In [31]:
len(df)

74

In [32]:
import pandas as pd

valid_json = df['json_data']
valid_json = valid_json[valid_json.apply(lambda x: isinstance(x, dict))]

all_keys = set()
for item in valid_json:
    all_keys.update(item.keys())

print(sorted(all_keys))

['notification', 'tmv', 'tmv3', 'work_order']


### TMV

In [33]:
import re
import numpy as np
import pandas as pd
from collections import Counter

na_like_values = ['NA', 'N/A', 'NULL', 'NONE', 'NAN']
pattern_na = re.compile(r'^\s*(NA|N/A|NULL|NaN)\s*$', re.IGNORECASE)

def is_na_value(value):
    """Check if a value is considered 'NA' based on the pattern."""
    if pd.isna(value) or value is None:
        return True
    if isinstance(value, str):
        return bool(pattern_na.match(value))
    return False

def clean_value(value):
    """Recursively converts 'NA' string values in dicts/lists to np.nan."""
    if isinstance(value, dict):
        return {k: clean_value(v) for k, v in value.items()}
    elif isinstance(value, list):
        return [clean_value(v) for v in value]
    elif isinstance(value, str) and is_na_value(value):
        return np.nan
    else:
        return value

def find_na_keys(d, parent=''):
    """Extracts flattened keys whose values are considered 'NA' (including np.nan)."""
    na_keys = []
    if isinstance(d, dict):
        for k, v in d.items():
            full_key = f"{parent}.{k}" if parent else k
            if isinstance(v, dict):
                na_keys.extend(find_na_keys(v, full_key))
            elif is_na_value(v): # Checks for string 'NA', None, and np.nan
                na_keys.append(full_key)
    return na_keys

def flattened_json(d):
    """
    Flattens a nested dictionary, specifically handling the 'tmv' structure.

    - Flattens the component list (keys 'a' through 'r') into distinct columns (e.g., 'a.component').
    - Flattens other nested dicts (e.g., 'technician', 'gearbox_oil_levels') into distinct columns (e.g., 'technician.date').
    - Handles 'completed?' by creating a unique column name for it.
    """
    flat_data = {}

    def _flatten(data, parent_key=''):
        if isinstance(data, dict):
            for k, v in data.items():
                new_key = f"{parent_key}.{k}" if parent_key else k

                if len(k) == 1 and 'a' <= k <= 'r' and isinstance(v, dict):
                    _flatten(v, new_key)
                
                elif isinstance(v, dict):
                    _flatten(v, new_key)
                else:
                    flat_data[new_key] = v

    _flatten(d)
    return flat_data

df_tmv = df.copy()

df_tmv['tmv'] = df_tmv['json_data'].apply(
    lambda x: x.get('tmv') if isinstance(x, dict) else None
)

df_tmv = df_tmv[df_tmv['tmv'].notnull()].copy()

df_tmv['workorder_id'] = df_tmv['workorder_id'].apply(lambda x: int(x) if pd.notnull(x) else None)

df_tmv['tmv'] = df_tmv['tmv'].apply(clean_value)

df_tmv['na_keys'] = df_tmv['tmv'].apply(find_na_keys)
na_counter = Counter(k for keys in df_tmv['na_keys'] for k in keys)
na_summary = pd.DataFrame(na_counter.items(), columns=['key', 'na_count']).sort_values('na_count', ascending=False)

flattened_rows = [flattened_json(r) for r in df_tmv['tmv'].fillna({})]
tmv = pd.DataFrame(flattened_rows) 
tmv.index = df_tmv.index
tmv['workorder_id'] = df_tmv['workorder_id'].astype('Int64')
tmv['filename'] = df_tmv['filename']

for i, col in enumerate(tmv.columns, start=1):
    print(f"{i:3d}. {col}")
    if col in ['workorder_id', 'filename']:
        continue
    valid_workorders = tmv.loc[tmv[col].notna(), 'workorder_id'].unique()
    if len(valid_workorders) > 0:
        workorder_list = ", ".join(map(str, valid_workorders))
        print(f"   Work Orders with data ({len(valid_workorders)}): {workorder_list}")
        print("-" * 80)


  1. electrical_appliance.inspection_type
   Work Orders with data (63): 4000583484, 4000588662, 4000597623, 4000619061, 4000626055, 4000638254, 4000643761, 4000661042, 4000686381, 4000693844, 4000453225, 4000501560, 4000506807, 4000512672, 4000517959, 4000535261, 4000545768, 4000557930, 4000564878, 4000577270, 4000583485, 4000588664, 4000597624, 4000602416, 4000606514, 4000612507, 4000638256, 4000661044, 4000673720, 4000680481, 4000686382, 4000457987, 4000619062, 4000479330, 4000626058, 4000648886, 4000654795, 4000693845, 4000668013, 4000643762, 4000630414, 4000673719, 4000570814, 4000551689, 4000680480, 4000654794, 4000668011, 4000630413, 4000606513, 4000612506, 4000577269, 4000602415, 4000539767, 4000528379, 4000523629, 4000495761, 4000490660, 4000484626, 4000473479, 4000467961, 4000462714, 4000447735, 4000442349
--------------------------------------------------------------------------------
  2. electrical_appliance.inspection_items.a.component
   Work Orders with data (63): 40005

In [34]:
import re
import numpy as np
import pandas as pd
from collections import Counter

na_like_values = ['NA', 'N/A', 'NULL', 'NONE', 'NAN']
pattern_na = re.compile(r'^\s*(NA|N/A|NULL|NaN)\s*$', re.IGNORECASE)

def is_na_value(value):
    """Check if a value is considered 'NA' based on the pattern."""
    if pd.isna(value) or value is None:
        return True
    if isinstance(value, str):
        return bool(pattern_na.match(value))
    return False

def clean_value(value):
    """Recursively converts 'NA' string values in dicts/lists to np.nan."""
    if isinstance(value, dict):
        return {k: clean_value(v) for k, v in value.items()}
    elif isinstance(value, list):
        return [clean_value(v) for v in value]
    elif isinstance(value, str) and is_na_value(value):
        return np.nan
    else:
        return value

def find_na_keys(d, parent=''):
    """Extracts flattened keys whose values are considered 'NA' (including np.nan)."""
    na_keys = []
    if isinstance(d, dict):
        for k, v in d.items():
            full_key = f"{parent}.{k}" if parent else k
            if isinstance(v, dict):
                na_keys.extend(find_na_keys(v, full_key))
            elif is_na_value(v): # Checks for string 'NA', None, and np.nan
                na_keys.append(full_key)
    return na_keys

def flattened_json(d):
    """
    Flattens a nested dictionary, specifically handling the 'tmv3' structure.

    - Flattens the component list (keys 'a' through 'r') into distinct columns (e.g., 'a.component').
    - Flattens other nested dicts (e.g., 'technician', 'gearbox_oil_levels') into distinct columns (e.g., 'technician.date').
    - Handles 'completed?' by creating a unique column name for it.
    """
    flat_data = {}

    def _flatten(data, parent_key=''):
        if isinstance(data, dict):
            for k, v in data.items():
                new_key = f"{parent_key}.{k}" if parent_key else k

                if len(k) == 1 and 'a' <= k <= 'r' and isinstance(v, dict):
                    _flatten(v, new_key)
                
                elif isinstance(v, dict):
                    _flatten(v, new_key)
                else:
                    flat_data[new_key] = v

    _flatten(d)
    return flat_data

df_tmv3 = df.copy()

df_tmv3['tmv3'] = df_tmv3['json_data'].apply(
    lambda x: x.get('tmv3') if isinstance(x, dict) else None
)

df_tmv3 = df_tmv3[df_tmv3['tmv3'].notnull()].copy()

df_tmv3['workorder_id'] = df_tmv3['workorder_id'].apply(lambda x: int(x) if pd.notnull(x) else None)

df_tmv3['tmv3'] = df_tmv3['tmv3'].apply(clean_value)

df_tmv3['na_keys'] = df_tmv3['tmv3'].apply(find_na_keys)
na_counter = Counter(k for keys in df_tmv3['na_keys'] for k in keys)
na_summary = pd.DataFrame(na_counter.items(), columns=['key', 'na_count']).sort_values('na_count', ascending=False)

flattened_rows = [flattened_json(r) for r in df_tmv3['tmv3'].fillna({})]
tmv3 = pd.DataFrame(flattened_rows) 
tmv3.index = df_tmv3.index
tmv3['workorder_id'] = df_tmv3['workorder_id'].astype('Int64')
tmv3['filename'] = df_tmv3['filename']

for i, col in enumerate(tmv.columns, start=1):
    print(f"{i:3d}. {col}")
    if col in ['workorder_id', 'filename']:
        continue
    valid_workorders = tmv.loc[tmv[col].notna(), 'workorder_id'].unique()
    if len(valid_workorders) > 0:
        workorder_list = ", ".join(map(str, valid_workorders))
        print(f"   Work Orders with data ({len(valid_workorders)}): {workorder_list}")
        print("-" * 80)


  1. electrical_appliance.inspection_type
   Work Orders with data (63): 4000583484, 4000588662, 4000597623, 4000619061, 4000626055, 4000638254, 4000643761, 4000661042, 4000686381, 4000693844, 4000453225, 4000501560, 4000506807, 4000512672, 4000517959, 4000535261, 4000545768, 4000557930, 4000564878, 4000577270, 4000583485, 4000588664, 4000597624, 4000602416, 4000606514, 4000612507, 4000638256, 4000661044, 4000673720, 4000680481, 4000686382, 4000457987, 4000619062, 4000479330, 4000626058, 4000648886, 4000654795, 4000693845, 4000668013, 4000643762, 4000630414, 4000673719, 4000570814, 4000551689, 4000680480, 4000654794, 4000668011, 4000630413, 4000606513, 4000612506, 4000577269, 4000602415, 4000539767, 4000528379, 4000523629, 4000495761, 4000490660, 4000484626, 4000473479, 4000467961, 4000462714, 4000447735, 4000442349
--------------------------------------------------------------------------------
  2. electrical_appliance.inspection_items.a.component
   Work Orders with data (63): 40005

In [35]:
import os
import pandas as pd
from openpyxl import Workbook

output_path = '../../output/train_maintenance_vehicle.xlsx'

os.makedirs(os.path.dirname(output_path), exist_ok=True)

if not os.path.exists(output_path):
    Workbook().save(output_path)

with pd.ExcelWriter(output_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    tmv.to_excel(writer, index=False, sheet_name='tmv'),
    tmv3.to_excel(writer, index=False, sheet_name='tmv3'),

print(f"✅ Exported successfully to '{output_path}' (replaced existing sheet)")


✅ Exported successfully to '../../output/train_maintenance_vehicle.xlsx' (replaced existing sheet)
